# Week 3 : RAG Pipeline

## 1. Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

## 2. Load dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/kaoutharhamdan/route-ma/export_final2.csv')
df.head(10)

In [ ]:
#droping the lines  that have problem >


## 3. Extract texts : our knowledge base for RAG

Transformation de format des données car FAISS + embeddings attend une LISTE de textes et non pas un DataFrame

In [ ]:
texts = df["infraction_desc"].tolist()

## 4. Embedding model : the semantic understanding layer

This model transforms text into vectors (numbers).

Example:

"STOP sign" → [0.12, -0.44, 0.88, ...]


**Why ? LLMs cannot search text directly, so we compare meaning using vectors**

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# explore more embedding models on : https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

## 5. Build FAISS index

- Step 1: Encode all texts => Each rule becomes a vector.
- Step 2: Create FAISS index => A structure optimized for similarity search.
- Step 3: Store vectors => FAISS now holds all knowledge.

----
What FAISS does conceptually:

- Instead of: “search words”
- I does : “search meaning”

In [ ]:
import faiss

embeddings = embedding_model.encode(texts)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index size:", index.ntotal)

## 6. Retrieval function

In [ ]:
def retrieve(query, k=3):
    # Convert query to vector
    query_vec = embedding_model.encode([query])
    # Search similar documents
    distances, indices = index.search(np.array(query_vec), k)
    # FAISS returns: closest vectors, their positions
    return [texts[i] for i in indices[0]]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "microsoft/Phi-4-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Pipeline setup
# This wraps the model into a simple function: input → prompt, output → text
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
def rag_answer(query):
    # Retrieve context
    docs = retrieve(query)
    # Build context : Combine retrieved documents into one block
    context = "\n".join(docs)
    # Build prompt
    # We give the LLM: background knowledge, user question
    prompt = f"""
Role: Moroccan traffic law expert.

Instructions:
- Answer strictly from the context.
- Do not add external knowledge.
- If missing info, say: "لا أملك معلومات كافية في السياق."
- Answer in Arabic only.
- Keep answers short and precise.

Context:
{context}

User Question:
{query}

Final Answer:
"""
    # Generate answer
    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3
    )

    return output[0]["generated_text"]

In [ ]:
print(rag_answer("ماذا تعني  رخصه دوليه للسياقه؟"))

In [ ]:
print(rag_answer("متى يجب تشغيل أضواء السيارة؟"))

In [ ]:
print(rag_answer("ما هي شروط الحصول على رخصة السياقة في المغرب؟"))

### 9. Chat interface

In [ ]:
def chat_rag():
    print("🚗 RAG Code de la Route (tape 'exit' pour quitter)\n")
    
    while True:
        query = input(" Question: ")
        
        if query.lower() == "exit":
            print("Fin du chatbot")
            break
        
        answer = rag_answer(query)
        
        print("\nRéponse:\n", answer)
        print("\n" + "-"*50 + "\n")

chat_rag()